In [ ]:
import pandas as pd

from src import PROJECT_DIR, logging
from src import utils as src_utils

from discovery_utils.getters import crunchbase
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts,
    google,
    google_slides,
)

PROJECT_NAME = src_utils.PROJECT_NAME
OUTPUT_DIR = src_utils.OUTPUT_DIR

import importlib;
importlib.reload(src_utils);
importlib.reload(charts)
importlib.reload(google);
importlib.reload(google_slides);

In [ ]:
CB = crunchbase.CrunchbaseGetter()

In [ ]:
def get_ids_from_config(config_name):
    config = src_utils.get_config_dict(config_name)
    selected_df = src_utils.get_companies_from_config(CB, config)

    selected_texts_df = CB.get_organisation_text(selected_df)

    relevant_check_df = pd.read_json(src_utils.OUTPUT_DIR / f"llm_check_MS_{config_name}.jsonl", lines=True)
    relevant_checked_df = (
        selected_df
        .merge(relevant_check_df[['id', 'is_relevant']], left_on='id', right_on='id', how='left')
        .merge(selected_texts_df[['id', 'text']], left_on='id', right_on='id', how='left')
    )
    matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()

    return matching_ids


In [ ]:
FUNDING_ROUND_TYPES = ["angel", "pre_seed", "seed", "series_a", "series_b"]

In [ ]:
def produce_stats(CB, matching_ids: list[str], category_name: str) -> None:
    """ Produce stats for the companies and output charts """ 
    # Check companies by querying ids
    matchings_orgs_df = CB.organisations_enriched.query("id in @matching_ids")

    # Get the funding rounds for the matching companies
    funding_rounds_df = (
        CB.select_funding_rounds(org_ids=matching_ids, funding_round_types=FUNDING_ROUND_TYPES)
    )

    # organise investors by each funding round
    investors_df = (
        CB.funding_rounds_enriched
        .query("funding_round_id in @funding_rounds_df.funding_round_id")
        .groupby("funding_round_id")
        .agg(investor_name=("investor_name", list))
        .reset_index()
    )

    funding_rounds_df = (
        funding_rounds_df
        .drop(columns=["investor_name"])
        .merge(investors_df, on="funding_round_id", how="left")
    )

    # generate basic time series
    ts_df = analysis_crunchbase.get_timeseries(
        matchings_orgs_df, 
        funding_rounds_df, 
        period='year', 
        min_year=2014, 
        max_year=2025
    )
    growth_rates = analysis.smoothed_growth(ts_df, year_start=2020, year_end=2024)
    growth_rates_df = pd.DataFrame(growth_rates, columns=[category_name]).T.reset_index().rename(columns={'index': 'theme'})  

    # Let's look into breakdown of deal types
    # deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(funding_rounds_df, 2014, 2025)
    aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(funding_rounds_df)

    # IPOs and acquisitions
    ipos_df = CB.ipos.query("org_id in @matching_ids")
    acquisitions_df = CB.acquisitions.query("acquiree_id in @matching_ids")

    if len(ipos_df) > 0:
        ipos_df.to_csv(OUTPUT_DIR / f"ipos_{category_name}.csv", index=False)

    if len(acquisitions_df) > 0:
        acquisitions_df.to_csv(OUTPUT_DIR / f"acquisitions_{category_name}.csv", index=False)
        

    # Fig variables
    prefix = f"{OUTPUT_DIR}/charts/{category_name}_"
    _scale = 2

    # Investment amounts (total)
    fig = charts.ts_bar(
        ts_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
    chart_filename = f"{prefix}raised_amount.png"
    fig.save(chart_filename, scale_factor=_scale)

    # Number of companies
    fig = charts.ts_bar(
        ts_df,
        variable='n_orgs_founded',
        variable_title="Number of companies founded",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of founded {category_name} companies")
    chart_filename = f"{prefix}no_of_companies.png"
    fig.save(chart_filename, scale_factor=_scale)    

    # Investment amounts (by type)
    investment_types_fig = analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)
    investment_types_fig = charts.configure_plots(investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}")
    investment_types_chart_filename = f"{prefix}investment_types.png"
    investment_types_fig.save(investment_types_chart_filename, scale_factor=_scale)

    # Investment counts (by type)
    investment_types_counts_fig = analysis_crunchbase.chart_investment_types_counts(aggregated_funding_types_df)
    investment_types_counts_fig = charts.configure_plots(investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}")
    investment_types_counts_chart_filename = f"{prefix}investment_types_counts.png"
    investment_types_counts_fig.save(investment_types_counts_chart_filename, scale_factor=_scale)

    # deal_sizes_fig = analysis_crunchbase.chart_deal_sizes(deals_df)
    # deal_sizes_fig = charts.configure_plots(deal_sizes_fig, chart_title=f"Deal sizes for {category_name}")
    # deal_sizes_chart_filename = f"{prefix}deal_sizes.png"
    # deal_sizes_fig.save(deal_sizes_chart_filename, scale_factor=_scale)

    # deal_sizes_counts_fig = analysis_crunchbase.chart_deal_sizes_counts(deal_counts_df)
    # deal_sizes_counts_fig = charts.configure_plots(deal_sizes_counts_fig, chart_title=f"Number of deals by size for {category_name}")
    # deal_sizes_counts_chart_filename = f"{prefix}deal_sizes_counts.png"
    # deal_sizes_counts_fig.save(deal_sizes_counts_chart_filename, scale_factor=_scale)

    return ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df


In [ ]:
all_ts_df = []
all_ipos_df = []
all_acquisitions_df = []
all_growth_rates = []
all_export_df = []
all_funding_rounds_df = []
all_orgs = []

In [ ]:
for config_name in src_utils.CONFIG_NAMES:
    logging.info(f"Processing {config_name}")
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    matching_ids = get_ids_from_config(config_name)
    ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df = produce_stats(CB, matching_ids, category_name)

    all_ts_df.append(ts_df.assign(theme=category_name))
    all_growth_rates.append(growth_rates_df)
    all_ipos_df.append(ipos_df.assign(theme=category_name))
    all_acquisitions_df.append(acquisitions_df.assign(theme=category_name))  
    all_funding_rounds_df.append(funding_rounds_df.assign(theme=category_name))
    all_orgs.append(matchings_orgs_df.assign(theme=category_name))    

In [ ]:
for config_name, category in src_utils.CB_CATEGORIES.items():
    logging.info(f"Processing {config_name}")
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    matching_ids = CB.get_companies_in_categories([category], "narrow").id.to_list()
    ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df = produce_stats(CB, matching_ids, category_name)

    all_ts_df.append(ts_df.assign(theme=category_name))
    all_growth_rates.append(growth_rates_df)
    all_ipos_df.append(ipos_df.assign(theme=category_name))
    all_acquisitions_df.append(acquisitions_df.assign(theme=category_name))   
    all_funding_rounds_df.append(funding_rounds_df.assign(theme=category_name))
    all_orgs.append(matchings_orgs_df.assign(theme=category_name))   

In [ ]:
try:
    all_ipos_df = pd.concat(all_ipos_df, ignore_index=True).to_csv(OUTPUT_DIR / "all_ipos.csv", index=False)
    all_acquisitions_df = pd.concat(all_acquisitions_df, ignore_index=True).to_csv(OUTPUT_DIR / "all_acquisitions.csv", index=False)
except:
    pass

all_growth_rates_df = pd.concat(all_growth_rates, ignore_index=True)
all_growth_rates_df.to_csv(OUTPUT_DIR / "growth_rates.csv", index=False)

all_ts_df = pd.concat(all_ts_df, ignore_index=True)
all_ts_df.to_csv(OUTPUT_DIR / "all_ts_df.csv", index=False)

all_funding_rounds_df = pd.concat(all_funding_rounds_df, ignore_index=True)
all_funding_rounds_df.to_csv(OUTPUT_DIR / "all_funding_rounds.csv", index=False)

all_orgs = pd.concat(all_orgs, ignore_index=True)
all_orgs.to_csv(OUTPUT_DIR / "all_orgs.csv", index=False)


## Low carbon heating - standalone

In [ ]:
low_carbon_heating_configs = [
    "biomass_heating",    
    "district_heating",
    "geothermal_energy",
    "heat_pumps",
    "heat_storage",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
]

In [ ]:
ids = []
for config in low_carbon_heating_configs:
    ids.extend(get_ids_from_config(config))
ids = list(set(ids))

In [ ]:
ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df = produce_stats(CB, ids, "Low-Carbon Heating")

In [ ]:
len(funding_rounds_df.query("announced_on_date > '2013-12-31'"))

In [ ]:
len(ids)

In [ ]:
growth_rates_df

## Upload to GSlides

In [ ]:
# ", ".join([s.capitalize() for s in FUNDING_ROUND_TYPES]).replace("_", " ")
FUNDING_ROUND_TYPES_STR = "Angel, Pre-seed, Seed, Series A, Series B"

In [ ]:
PRESENTATION_ID = "1dfKE9j5Z1wegJjnGRc7Eg__cyndi1MoFcOV3IlzmjIM"
TEMPLATE_SLIDE = "g33a945c2053_0_666"

In [ ]:
category = "Bioenergy"
round(all_growth_rates_df.query("theme == @category")['n_rounds'].iloc[0],0)

In [ ]:
import os
importlib.reload(os)
importlib.reload(google)
importlib.reload(google_slides)

In [ ]:
gdrive_service = google_slides.get_drive_service()
gslides_service = google_slides.get_slides_service()

In [ ]:
category_names = []
for config_name in src_utils.CONFIG_NAMES:
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    category_names.append(category_name)

for config_name, category in src_utils.CB_CATEGORIES.items():
    config = src_utils.get_config_dict(config_name)
    category_name = config["search_recipe"]["category_name"]
    category_names.append(category_name)

category_names = list(reversed(category_names))
category_names

In [ ]:
# for config_name, category in src_utils.CB_CATEGORIES.items():
all_file_ids = []

for category in category_names:
    logging.info(f"Processing {category}")
    all_requests = []

    slide_id = category.lower().replace(" ", "_")

    # Investment amounts
    fig_path = f"{OUTPUT_DIR}/charts/{category}_investment_types.png"
    file_id, image_url = google.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_funding = {
        "chart_type": "funding",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates_df.query("theme == @category")['raised_amount_gbp_total'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": 41,
    }

    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_funding",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_funding)
        )
        .slide_request()
    )

    # Number of investment rounds
    fig_path = f"{OUTPUT_DIR}/charts/{category}_investment_types_counts.png"
    file_id, image_url = google.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_rounds = {
        "chart_type": "number_of_rounds",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates_df.query("theme == @category")['n_rounds'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": -1.2,
    }

    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_rounds",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_rounds)
        )
        .slide_request()
    )

    # Number of companies
    slide_id = category.lower().replace(" ", "_")
    fig_path = f"{OUTPUT_DIR}/charts/{category}_no_of_companies.png"
    file_id, image_url = google.upload_image_to_drive(gdrive_service, fig_path)
    all_file_ids.append(file_id)

    text_new_companies = {
        "chart_type": "number_of_new_companies",
        "investment_types": FUNDING_ROUND_TYPES_STR,
        "category": f"{category}",
        "n_companies": len(all_orgs.query("theme == @category").drop_duplicates("id")),
        "n_rounds": len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id")),
        "year_start": 2014,
        "year_end": 2025,
        "growth": round(all_growth_rates_df.query("theme == @category")['n_orgs_founded'].iloc[0],0),
        "growth_start": 2020,
        "growth_end": 2024,
        "baseline_growth": -71,
    }


    all_requests += (
        google_slides
        .MissionStudioTemplate(
            template_id = TEMPLATE_SLIDE,
            slide_id = slide_id + "_companies",
            image_url = image_url,
            heading_text = category,
            details_text = google_slides.text_investment(**text_new_companies)
        )
        .slide_request()
    )

    response = (
        gslides_service
        .presentations()
        .batchUpdate(presentationId=PRESENTATION_ID, body={"requests": all_requests})
        .execute()
    )  

In [ ]:
for file_id in all_file_ids:
    try:
        google.delete_file_from_drive(gdrive_service, file_id)      
    except Exception as e:
        logging.error(f"Error deleting file: {e}")

In [ ]:
# for config_name, category in src_utils.CB_CATEGORIES.items():
all_file_ids = []

for category in category_names:
    print(category)
    print(len(all_funding_rounds_df.query("theme == @category").drop_duplicates("funding_round_id").query("announced_on_date > '2013-12-31'")))

In [ ]:
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    "theme",
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_name",
]

In [ ]:
cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    "theme",    
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    "region_nesta",
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
]

In [ ]:
_all_funding_rounds_df = (
    all_funding_rounds_df
    .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_funding_rounds]

In [ ]:
_all_orgs = (
    all_orgs
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_companies]

In [ ]:
all_ipos_df = pd.read_csv(OUTPUT_DIR / "all_ipos.csv")
all_acquisitions_df = pd.read_csv(OUTPUT_DIR / "all_acquisitions.csv")

In [ ]:
sheet_id = "1dh46jwXv-es5b-gnvsRxzHYDtFOrwn4pWi3DWs2PGbo"

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_funding": _all_funding_rounds_df})
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_ipos": all_ipos_df})
google.format_gsheet(sheet_id, "crunchbase_ipos", freeze_cols=0)

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_aquisitions": all_acquisitions_df})
google.format_gsheet(sheet_id, "crunchbase_aquisitions", freeze_cols=0)

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_companies": _all_orgs})
google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)